In [1]:
import tensorflow as tf
import cv2
import numpy as np
import albumentations as A
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

# Cek MobileNetV2 bisa diload
model = tf.keras.applications.MobileNetV2(weights='imagenet')
print("MobileNetV2 berhasil diload!")

TensorFlow version: 2.21.0
GPU tersedia: []
MobileNetV2 berhasil diload!


## 1. Load Dataset Menggunakan `image_dataset_from_directory` karena data (dan augmentasinya) sudah tersedia di folder masing-masing.

In [2]:
import os

dataset_dir = r"c:\Users\Tristan\Downloads\cleaned_dataset"
train_dir = os.path.join(dataset_dir, 'train')
val_dir = os.path.join(dataset_dir, 'val')
test_dir = os.path.join(dataset_dir, 'test')

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

class_names = train_dataset.class_names
print("Class names:", class_names)


Found 18000 files belonging to 6 classes.
Found 2040 files belonging to 6 classes.
Found 2040 files belonging to 6 classes.
Class names: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']


## 2. Prefetching & Preprocessing\nMeningkatkan performa I/O dataset dan menerapkan preprocessing bawaan MobileNetV2.

In [3]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input


## 3. Membangun Arsitektur Model

In [4]:
# Buat base model MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

# Freeze base model (kita tidak melatih layer dasar terlebih dahulu)
base_model.trainable = False

# Layer Arsitektur Klasifikasi Kustom
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = preprocess_input(inputs) # Preprocessing spesifik MobileNetV2
x = base_model(x, training=False) # Pastikan BatchNorm layer jalan di inference mode
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(6, activation='softmax')(x) # 6 kelas buah

model = tf.keras.Model(inputs, outputs)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 4. Kompilasi & Pelatihan Model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# custom callback
class TargetCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('accuracy') >= 0.95 and logs.get('val_accuracy') >= 0.98):
            print("menghentikan proses training")
            self.model.stop_training = True

my_callback = TargetCallback()

# Setup Callbacks
callbacks = [
    my_callback,
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_mobilenet_model.keras', monitor='val_accuracy', save_best_only=True)
]

# Jalankan training
initial_epochs = 10

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=initial_epochs,
    callbacks=callbacks
)


Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 122s 213ms/step - accuracy: 0.7371 - loss: 0.7814 - val_accuracy: 0.9373 - val_loss: 0.2861
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 109s 194ms/step - accuracy: 0.9030 - loss: 0.3280 - val_accuracy: 0.9637 - val_loss: 0.1702
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 156s 278ms/step - accuracy: 0.9257 - loss: 0.2447 - val_accuracy: 0.9721 - val_loss: 0.1297
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 255s 371ms/step - accuracy: 0.9376 - loss: 0.2042 - val_accuracy: 0.9765 - val_loss: 0.1057
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 189s 335ms/step - accuracy: 0.9416 - loss: 0.1807 - val_accuracy: 0.9789 - val_loss: 0.0919
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 231s 387ms/step - accuracy: 0.9480 - loss: 0.1650 - val_accuracy: 0.9804 - val_loss: 0.0821
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 280s 418ms/step - accuracy: 0.9527 - loss: 0.1515 - val_accuracy: 0.9843 - val_loss: 0.0733
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 228s 357ms/step - accuracy: 0.9548 -

## 5. Evaluasi & Ekspor ke .keras

In [6]:
# Evaluasi di data testing

loss, accuracy = model.evaluate(test_dataset)
print(f'Test accuracy: {accuracy:.4f}')

# Simpan model untuk dikonversi ke TFJS
model_save_path = 'mobilenet_fruit_model.keras'
model.save(model_save_path)
print(f"Model berhasil disimpan di: {model_save_path}")


 2/64 ━━━━━━━━━━━━━━━━━━━━ 16s 270ms/step - accuracy: 1.0000 - loss: 0.0361

64/64 ━━━━━━━━━━━━━━━━━━━━ 20s 312ms/step - accuracy: 0.9887 - loss: 0.0488
Test accuracy: 0.9887
Model berhasil disimpan di: mobilenet_fruit_model.keras
